# 기업개요 CSV 4개 통합

1~4페이지를 순서대로 합쳐 `data/기업개요_통합.csv`에 저장합니다. 법인등록번호 등의 앞자리 0을 보존하도록 모든 값을 문자열로 처리하고, 중복 제거 없이 모든 행을 유지합니다. 원본 CSV는 변경하지 않습니다.

In [1]:
import csv
from pathlib import Path

# 프로젝트 루트 또는 하위 폴더에서 실행할 수 있습니다.
project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / '금융위원회_기업기본정보_수집').is_dir()),
    None,
)
if project_root is None:
    raise FileNotFoundError('프로젝트의 금융위원회_기업기본정보_수집 폴더를 찾을 수 없습니다.')

source_dir = project_root / '금융위원회_기업기본정보_수집'
source_files = [
    source_dir / '기업개요_1페이지_10000건.csv',
    source_dir / '기업개요_페이지_2.csv',
    source_dir / '기업개요_페이지_3.csv',
    source_dir / '기업개요_페이지_4.csv',
]
output_file = project_root / 'data' / '기업개요_통합.csv'

# 저장 전에 파일 4개의 컬럼 구성과 행 구조를 확인합니다.
columns = None
merged_rows = []
for source_file in source_files:
    with source_file.open('r', encoding='utf-8-sig', newline='') as file:
        reader = csv.reader(file)
        header = next(reader, None)
        if not header:
            raise ValueError(f'헤더가 없습니다: {source_file.name}')
        if columns is None:
            columns = header
        elif header != columns:
            raise ValueError(f'컬럼 구성이 다릅니다: {source_file.name}')

        row_count = 0
        for row in reader:
            if not row:
                continue
            if len(row) != len(columns):
                raise ValueError(f'행의 컬럼 수가 다릅니다: {source_file.name}, {reader.line_num}행')
            merged_rows.append(row)
            row_count += 1
        print(f'{source_file.name}: {row_count:,}행')

output_file.parent.mkdir(parents=True, exist_ok=True)
with output_file.open('w', encoding='utf-8-sig', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(columns)
    writer.writerows(merged_rows)

# 저장된 데이터가 합친 원본 데이터와 같은지 확인합니다.
with output_file.open('r', encoding='utf-8-sig', newline='') as file:
    reader = csv.reader(file)
    assert next(reader) == columns
    assert list(reader) == merged_rows, '저장된 데이터가 원본과 다릅니다.'

print(f'통합 완료: {len(merged_rows):,}행 × {len(columns)}개 컬럼')
print(f'저장 위치: {output_file}')


기업개요_1페이지_10000건.csv: 10,000행
기업개요_페이지_2.csv: 10,000행
기업개요_페이지_3.csv: 10,000행
기업개요_페이지_4.csv: 10,000행
통합 완료: 40,000행 × 37개 컬럼
저장 위치: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\기업개요_통합.csv


In [5]:
import pandas as pd

df=pd.read_csv('기업개요_통합.csv')
df.columns

C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\3553099711.py:3: DtypeWarning: Columns (0: enpKrxLstgDt, 1: enpKrxLstgAbolDt) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('기업개요_통합.csv')


Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm',
       'corpRegMrktDcd', 'corpRegMrktDcdNm', 'corpDcd', 'corpDcdNm', 'bzno',
       'enpOzpno', 'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'enpFxno',
       'sicNm', 'enpEstbDt', 'enpStacMm', 'enpXchgLstgDt', 'enpXchgLstgAbolDt',
       'enpKosdaqLstgDt', 'enpKosdaqLstgAbolDt', 'enpKrxLstgDt',
       'enpKrxLstgAbolDt', 'smenpYn', 'enpMntrBnkNm', 'enpEmpeCnt',
       'empeAvgCnwkTermCtt', 'enpPn1AvgSlryAmt', 'actnAudpnNm',
       'audtRptOpnnCtt', 'enpMainBizNm', 'fssCorpUnqNo', 'fssCorpChgDtm',
       'fstOpegDt', 'lastOpegDt'],
      dtype='str')

In [6]:
df1=df.drop(columns=['corpRegMrktDcd','corpRegMrktDcdNm','corpDcd','corpDcdNm','enpOzpno','enpFxno','enpStacMm','enpKosdaqLstgDt','enpKosdaqLstgAbolDt','enpKrxLstgDt','enpKrxLstgAbolDt','smenpYn','enpMntrBnkNm','enpPn1AvgSlryAmt','actnAudpnNm','fssCorpUnqNo','fssCorpChgDtm','fstOpegDt'])
df1.columns

Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm', 'bzno',
       'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'sicNm', 'enpEstbDt',
       'enpXchgLstgDt', 'enpXchgLstgAbolDt', 'enpEmpeCnt',
       'empeAvgCnwkTermCtt', 'audtRptOpnnCtt', 'enpMainBizNm', 'lastOpegDt'],
      dtype='str')

In [8]:
df1.isna().sum()

crno                      0
corpNm                    0
corpEnsnNm            13060
enpPbanCmpyNm         19116
enpRprFnm              9634
bzno                   9937
enpBsadr               7458
enpDtadr              22743
enpHmpgUrl            24018
enpTlno                7307
sicNm                 38295
enpEstbDt             10819
enpXchgLstgDt         35356
enpXchgLstgAbolDt     39521
enpEmpeCnt                0
empeAvgCnwkTermCtt    32561
audtRptOpnnCtt        34285
enpMainBizNm          39738
lastOpegDt                0
dtype: int64

In [10]:
df1.info()

<class 'pandas.DataFrame'>
RangeIndex: 40000 entries, 0 to 39999
Data columns (total 19 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   crno                40000 non-null  int64  
 1   corpNm              40000 non-null  str    
 2   corpEnsnNm          26940 non-null  str    
 3   enpPbanCmpyNm       20884 non-null  str    
 4   enpRprFnm           30366 non-null  str    
 5   bzno                30063 non-null  float64
 6   enpBsadr            32542 non-null  str    
 7   enpDtadr            17257 non-null  str    
 8   enpHmpgUrl          15982 non-null  str    
 9   enpTlno             32693 non-null  str    
 10  sicNm               1705 non-null   str    
 11  enpEstbDt           29181 non-null  float64
 12  enpXchgLstgDt       4644 non-null   str    
 13  enpXchgLstgAbolDt   479 non-null    str    
 14  enpEmpeCnt          40000 non-null  int64  
 15  empeAvgCnwkTermCtt  7439 non-null   str    
 16  audtRptOpnnCtt 

In [18]:
df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')

C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\1953989232.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')
C:\Users\Playdata\AppData\Local\Temp\ipykernel_7532\1953989232.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df1[['enpXchgLstgDt','enpXchgLstgAbolDt']] = df1[['enpXchgLstgDt','enpXchgLstgAbolDt']].apply(pd.to_datetime, errors='coerce')


In [19]:
df1[['enpXchgLstgDt','enpXchgLstgAbolDt']]

,enpXchgLstgDt,enpXchgLstgAbolDt
0,NaT,NaT
1,NaT,NaT
2,NaT,NaT
3,NaT,NaT
4,NaT,NaT
...,...,...
39995,NaT,NaT
39996,NaT,NaT
39997,NaT,NaT
39998,NaT,NaT


In [ ]:
df1[df1['enpXchgLstgDt']<df1['enpXchgLstgAbolDt']]

df1[(df1['enpXchgLstgDt'].notna()) & (df1['crno']==1101110002818)]
df1[df1['enpXchgLstgDt'].dt.year > 2026]

# df1[df1['crno']==1101110002818]

df[df['crno']==1101110003262]

282    56/03/03
283    56/03/03
284    56/03/03
285    56/03/03
286    56/03/03
287         NaN
288    56/03/03
289    56/03/03
290    56/03/03
291    56/03/03
292    56/03/03
293    56/03/03
294    56/03/03
295    56/03/03
296    56/03/03
Name: enpXchgLstgDt, dtype: str

In [49]:
mask = df1['enpXchgLstgDt'].dt.year > 2026
df1.loc[mask, 'enpXchgLstgDt'] = df1.loc[mask, 'enpXchgLstgDt'] - pd.DateOffset(years=100)
mask2 = df1['enpXchgLstgAbolDt'].dt.year > 2026
df1.loc[mask2, 'enpXchgLstgAbolDt'] = df1.loc[mask2, 'enpXchgLstgAbolDt'] - pd.DateOffset(years=100)

In [54]:

df1=df1.drop(columns=['enpXchgLstgDt','enpXchgLstgAbolDt'])
df1



,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,0,SAMPO FUND MANAGEMENT LTD/MANDATUM EMERGING,SAMPO FUND MANAGEMENT LTD/MANDATUM EMERGING,SAMPOFUNDMANAGEMENTLTD/MANDATUMEMERGING,KIMMO LAAKSONEN,NaN,서울특별시 종로구 신문로2가 시티빌딩,NaN,NaN,0220041331,NaN,19990901.0,0,NaN,NaN,NaN,20260914,0
1,0,"ARMOR QUALIFIED, LP","ARMOR QUALIFIED, LP","ARMORQUALIFIED,LP",KRISTINE GUARNERI,NaN,서울특별시 중구 다동39번지,NaN,NaN,212-808-3721,NaN,20060120.0,0,NaN,NaN,NaN,20260914,0
2,0,리만 브라더스,LEHMAN BROTHERS INC,리만브라더스,"RICHARD S.FUND,JR.",NaN,"745 Seventh Avenue New York, NY 10019",NaN,www.lehman.com,1-212-526-7000,NaN,19650121.0,0,NaN,NaN,NaN,20260914,0
3,0,Citigroup Financial Products Inc.,Citigroup Financial Products Inc.,CitigroupFinancialProductsInc.,Robert Druskin,NaN,서울특별시 중구 다동,NaN,NaN,212-816-5605,NaN,19840706.0,0,NaN,NaN,NaN,20260914,0
4,0,"FIREBIRD GLOBAL MASTER FUND, LTD","FIREBIRD GLOBAL MASTER FUND, LTD","FIREBIRDGLOBALMASTERFUND,LTD",JAMES PASSIN,NaN,서울특별시 종로구 신문로2가 시티빌딩,NaN,NaN,0220041331,NaN,20030508.0,0,NaN,NaN,NaN,20260914,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39995,1101110897269,(주)홍익애이디넷,NaN,NaN,NaN,NaN,전라남도 나주시,동수농공단지길 62-40 ((운곡동) 운곡동),NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,20250318,0
39996,1101110897269,(주)홍익애이디넷,HONG IK,NaN,정원찰,1.088144e+09,서울 강남구 역삼동,603-3 타비쉬빌딩 6층,NaN,82-02-554-0951,NaN,19921022.0,10,NaN,NaN,NaN,20200517,0
39997,1101110897300,(주)명성건축ENG.,MYOUNG SUNG ARCHITECT,NaN,이강억,1.138126e+09,서울 구로구 구로6동,95-6,NaN,82-02-852-8521,NaN,19921022.0,0,NaN,NaN,NaN,20200517,0
39998,1101110897409,(주)연세스포츠센터리즈마트,NaN,NaN,서태순,1.108143e+09,서울 서대문구 홍제동,158-33,NaN,82-02-391-3500,NaN,19991015.0,0,NaN,NaN,NaN,20200517,0


Index(['crno', 'corpNm', 'corpEnsnNm', 'enpPbanCmpyNm', 'enpRprFnm', 'bzno',
       'enpBsadr', 'enpDtadr', 'enpHmpgUrl', 'enpTlno', 'sicNm', 'enpEstbDt',
       'enpEmpeCnt', 'empeAvgCnwkTermCtt', 'audtRptOpnnCtt', 'enpMainBizNm',
       'lastOpegDt', '상장여부'],
      dtype='str')

In [ ]:
import pandas as pd
from pathlib import Path

csv_path = Path('기업개요전처리.csv')

# crno는 계산용 숫자가 아니라 식별자이므로 문자열로 읽습니다.
df = pd.read_csv(csv_path, dtype={'crno': 'string'})

# CSV 저장 과정에서 생긴 불필요한 인덱스 열이 있으면 제거합니다.
df = df.loc[:, ~df.columns.str.startswith('Unnamed:')]

before_count = len(df)
crno_is_zero = df['crno'].fillna('').str.strip().eq('0')
df = df.loc[~crno_is_zero].reset_index(drop=True)

print(f'삭제한 행 수: {before_count - len(df):,}')
print(f'남은 행 수: {len(df):,}')
assert not df['crno'].fillna('').str.strip().eq('0').any()


삭제한 행 수: 24
남은 행 수: 39,976


,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,4108280449,두류야외음악당지역주택조합,NaN,NaN,석자은,4.108280e+09,대구광역시 달서구 달구벌대로 1666 (감삼스퀘어),", 502호,503호",NaN,0535577600,건설업(41-42),NaN,4,NaN,NaN,NaN,20260914,0
1,4108280449,두류야외음악당지역주택조합,NaN,NaN,석자은,4.108280e+09,대구광역시 달서구 달구벌대로 1666 (감삼스퀘어),", 502호,503호",NaN,0535577600,NaN,NaN,4,NaN,NaN,NaN,20250331,0
2,4178401361,(주)라이프엣그,life,NaN,마사추쿠히로시,4.178401e+09,전남 여수시 덕충동,100 (덕충안길),NaN,82-061-6282-3940,NaN,20120512.0,0,NaN,NaN,NaN,20200517,0
3,13000001504,솔라원오호 주식회사,solarone5,솔라원오호,정재훈,6.838804e+09,"전북특별자치도 고창군 흥덕면 부안로 148 (치룡리, 쏠라파크태양광발전소)",NaN,NaN,01*********,NaN,20251022.0,0,NaN,NaN,NaN,20260422,0
4,105020209750,(유)경응,KYUNG EUNG,NaN,강용구,1.050202e+08,대구 달서구 송현동,1990-11,NaN,82-053-656-1341,NaN,NaN,0,NaN,NaN,NaN,20200517,0


In [64]:
df[df['corpNm']=='(주)유수홀딩스']

,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
258,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,16년 2개월,적정의견,NaN,20260914,1
259,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,16년 2개월,NaN,NaN,20260323,1
260,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260322,1
261,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260310,1
262,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)","서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,NaN,NaN,20260101,1
264,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,24,14년 9개월,적정의견,NaN,20251007,1
265,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,14년5개월,적정의견,NaN,20250320,1
266,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,23,14년5개월,NaN,NaN,20240320,1
267,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,22,14년 6개월,적정의견,NaN,20240319,1
268,1101110003262,(주)유수홀딩스,"EUSU HOLDINGS CO., LTD.",유수홀딩스,송영규,1.168136e+09,"서울특별시 영등포구 여의나루로 60 9층 (여의도동, 포스트타워)",NaN,www.eusu-holdings.com,02-6716-3000,NaN,19500101.0,25,12년 1개월,적정의견,NaN,20230320,1


In [65]:
# crno가 0인 행을 제외하고, crno별 lastOpegDt가 가장 최신인 행만 남깁니다.
df = df[df['crno'].fillna('').astype('string').str.strip().ne('0')].copy()

df['_lastOpegDt'] = pd.to_datetime(
    df['lastOpegDt'].astype('string').str.strip(),
    format='%Y%m%d',
    errors='coerce'
)

# 날짜가 같은 중복 행은 현재 순서상 첫 번째 행을 남깁니다.
before_count = len(df)
df = (
    df.sort_values(
        ['crno', '_lastOpegDt'],
        ascending=[True, False],
        na_position='last',
        kind='stable'
    )
    .drop_duplicates(subset='crno', keep='first')
    .drop(columns='_lastOpegDt')
    .reset_index(drop=True)
)

print(f'삭제한 중복 행 수: {before_count - len(df):,}')
print(f'최신 행만 남긴 데이터 크기: {df.shape}')
assert not df['crno'].duplicated().any()


삭제한 중복 행 수: 22,587
최신 행만 남긴 데이터 크기: (17389, 18)


In [68]:
df.head()

,crno,corpNm,corpEnsnNm,enpPbanCmpyNm,enpRprFnm,bzno,enpBsadr,enpDtadr,enpHmpgUrl,enpTlno,sicNm,enpEstbDt,enpEmpeCnt,empeAvgCnwkTermCtt,audtRptOpnnCtt,enpMainBizNm,lastOpegDt,상장여부
0,1001110851984,삼성금은 주식회사,"samsunggold&silver co., ltd",삼성금은,이 규석,2.088117e+09,서울특별시 종로구 봉익동 141-1 세화빌딩 211호,NaN,www.samsunggold.com,02-764-2869,NaN,19920523.0,0,NaN,NaN,NaN,20260914,0
1,1001116020799,이지앤스토리 주식회사,"EZNstory Co.,Ltd.",이지앤스토리,"유승민,조현철",1.608700e+09,"서울특별시 구로구 디지털로34길 43 1107호 (구로동, 코오롱싸이언스밸리1차)",NaN,www.eznstory.com,02-6952-3711,NaN,20160406.0,0,NaN,NaN,NaN,20240403,0
2,1001116253720,주식회사 파낙토스,Panaxtos Corp.,파낙토스,박병운,3.628601e+09,"서울특별시 송파구 올림픽로 342 (방이동, 아울타워) 15층",NaN,www.panaxtos.com,02-2051-1380,NaN,20161208.0,0,NaN,NaN,NaN,20231212,0
3,1001140001882,인듐코퍼레이션 코리아(유),"Indium Corporation (Korea)Co.,Ltd",인듐코퍼레이션코리아,그레고리 피에번스,3.018196e+09,"충청북도 청주시 흥덕구 직지대로436번길 24 (송정동, 인듐코퍼레이션)",NaN,www.indiumcom,07077805200,NaN,20080114.0,0,NaN,NaN,NaN,20250920,0
4,1008100969000,동해양조공업(주),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,NaN,NaN,NaN,20260914,0


In [70]:
df.to_csv('기업개요.csv')

In [ ]:
import pandas as pd
import json
import re
from pathlib import Path

input_path = Path(r'C:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\기업개요.csv')
output_path = input_path.with_name('기업개요_대표자정리.csv')

df_rep = pd.read_csv(input_path, dtype={'enpRprFnm': 'string'})
df_rep = df_rep.loc[:, ~df_rep.columns.str.startswith('Unnamed:')]

# 직함은 이름과 붙어 있거나 괄호 안에 있을 수 있으므로 직함만 제거합니다.
role_re = re.compile(
    r'(?<![가-힣])(?:각자\s*대표\s*이사|공동\s*대표\s*이사|단독\s*대표\s*이사|대표\s*집행\s*임원|대표\s*이사|대표이사|대표자|대표|회장|부회장|사장|집행\s*임원|이사)(?![가-힣])',
    flags=re.IGNORECASE,
)
role_word_re = re.compile(r'대표|이사|회장|사장|공동|각자|단독|집행임원')
count_re = re.compile(r'\s*외\s*\d+\s*(?:명|인)?|\s*외\s*$')

def keep_parenthetical(match):
    content = match.group(1).strip()
    # 직함·인원수 설명은 제거하고, 괄호 안의 실제 이름은 보존합니다.
    if role_word_re.search(content) or re.search(r'\d+\s*(?:명|인)', content):
        return ' '
    return f', {content}, '

def clean_representatives(value):
    if pd.isna(value):
        return []
    text = str(value).replace('\n', ' ').strip()
    text = re.sub(r'\(([^()]*)\)', keep_parenthetical, text)
    text = count_re.sub(' ', text)
    text = role_re.sub(' ', text)
    text = re.sub(r'\s+', ' ', text).strip(' ,')
    if not text:
        return []

    # 쉼표·슬래시·세미콜론·및을 대표자 구분자로 처리합니다.
    parts = re.split(r'\s*(?:,|/|;|·|및)\s*', text)
    names = []
    for part in parts:
        part = re.sub(r'\s+', ' ', part).strip(' ,')
        if not part:
            continue
        # 한국 이름은 성과 이름 사이 공백을 제거합니다.
        if re.fullmatch(r'[가-힣 ]+', part):
            tokens = part.split()
            if len(tokens) > 1 and all(re.fullmatch(r'[가-힣]{2,3}', token) for token in tokens):
                names.extend(tokens)
            else:
                names.append(''.join(tokens))
        else:
            names.append(part)

    return list(dict.fromkeys(names))

# df_rep에서는 실제 파이썬 리스트로 보관합니다.
df_rep['enpRprFnm'] = df_rep['enpRprFnm'].map(clean_representatives)

# CSV에는 리스트를 JSON 문자열로 저장해 재사용할 수 있게 합니다.
csv_df = df_rep.copy()
csv_df['enpRprFnm'] = csv_df['enpRprFnm'].map(lambda names: json.dumps(names, ensure_ascii=False))
csv_df.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'저장 위치: {output_path}')
print(f'데이터 크기: {df_rep.shape}')
print(df_rep[['enpRprFnm']].head())


In [72]:
import csv
import json
from pathlib import Path

input_path = Path('기업개요_대표자정리.csv')
output_path = input_path.with_suffix('.jsonl')

with input_path.open('r', encoding='utf-8-sig', newline='') as input_file, output_path.open('w', encoding='utf-8', newline='') as output_file:
    reader = csv.DictReader(input_file)
    for row in reader:
        # CSV에 JSON 문자열로 저장된 대표자 리스트를 실제 리스트로 복원합니다.
        row['enpRprFnm'] = json.loads(row['enpRprFnm'])
        # 빈 문자열은 JSON null로 저장합니다.
        row = {key: (None if value == '' else value) for key, value in row.items()}
        output_file.write(json.dumps(row, ensure_ascii=False) + '\n')

print(f'저장 위치: {output_path}')


저장 위치: 기업개요_대표자정리.jsonl


In [73]:
df=pd.read_csv("기업개요_대표자정리.csv")


In [75]:
len(df)

17389

In [74]:
df.isna().sum()

crno                      0
corpNm                    0
corpEnsnNm             7964
enpPbanCmpyNm         11281
enpRprFnm                 0
bzno                   6134
enpBsadr               5111
enpDtadr              11471
enpHmpgUrl            13551
enpTlno                6178
sicNm                 17360
enpEstbDt              7182
enpEmpeCnt                0
empeAvgCnwkTermCtt    16763
audtRptOpnnCtt        16811
enpMainBizNm          17382
lastOpegDt                0
상장여부                      0
dtype: int64

In [ ]:


('모기업','계열관계다','모기업')
('모기업','종속관계다','종속기업')
('모기업','위치해있다','지역')
('종속기업','위치해있다','지역')
('종속기업','관련있다','업종')






# 기업 관계 지식그래프 온톨로지

교안의 관계 시그니처, 트리플 추출, 시그니처 검증 흐름을 기업 데이터에 적용한다.

- ParentCompany: 기업개요 CSV의 crno를 식별자로 사용하는 모기업 노드
- SubsidiaryCompany: crno가 없으므로 정규화된 기업명과 주소를 전역 합성 키로 사용하는 노드
- Region: 기업 또는 종속기업의 지역 노드
- Industry: 기업의 업종 또는 종속기업 주요사업내용 노드

입력 파일은 data/clean/기업개요_최종.csv와 data/clean/모기업_계열사_종속기업_통합.csv이다. 기업개요에 존재하는 crno만 ParentCompany로 만들며, 같은 종속기업이 여러 모기업에 연결되면 하나의 SubsidiaryCompany 노드에 여러 HAS_SUBSIDIARY 관계를 연결한다.


In [1]:
from pathlib import Path
import html
import json
import re
import pandas as pd

CORP_FILENAME = "최종_기업개요.csv"
INTEGRATED_FILENAME = "최종_모기업_계열사_종속기업_통합.csv"
PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "data" / "clean" / CORP_FILENAME).exists()),
    Path.cwd(),
)
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
CORP_PATH = CLEAN_DIR / CORP_FILENAME
INTEGRATED_PATH = CLEAN_DIR / INTEGRATED_FILENAME

ONTOLOGY_SCHEMA = {
    "nodes": {
        "ParentCompany": {
            "description": "계열관계 또는 종속관계의 기준이 되는 법인 기업",
            "properties": {
                "crno": {"type":"string", "description":"법인등록번호"},
                "name": {"type":"string", "description":"기업명"},
                "address": {"type":"string", "description":"기업 주소"},
            },
        },
        "SubsidiaryCompany": {
            "description": "모기업과 종속관계를 가지는 기업",
            "properties": {
                "name": {"type":"string", "description":"종속기업명"},
                "name_norm": {"type":"string", "description":"기업명 매칭을 위한 정규화 이름"},
                "address": {"type":"string", "description":"종속기업 주소"},
                "business_content": {"type":"string", "description":"종속기업의 주요 사업 내용"},
                "domestic": {"type":"string", "description":"국내 또는 해외 구분"},
            },
        },
        "Region": {
            "description": "기업이 위치한 지역",
            "properties": {"name": {"type":"string", "description":"지역명"}},
        },
        "Industry": {
            "description": "기업이 영위하거나 관련된 업종",
            "properties": {"name": {"type":"string", "description":"업종명"}},
        },
    },
    "relationships": {
        "AFFILIATED_WITH": {
            "description": "두 모기업이 같은 기업집단 또는 계열 관계에 있음을 의미",
            "source":"ParentCompany", "target":"ParentCompany",
        },
        "HAS_SUBSIDIARY": {
            "description": "모기업이 해당 종속기업과 종속 관계에 있음을 의미",
            "source":"ParentCompany", "target":"SubsidiaryCompany",
        },
        "LOCATED_IN": {
            "description": "기업이 해당 지역에 위치함을 의미",
            "signatures":[
                {"source":"ParentCompany","target":"Region"},
                {"source":"SubsidiaryCompany","target":"Region"},
            ],
        },
        "IN_INDUSTRY": {
            "description": "기업이 해당 업종을 영위하거나 관련됨을 의미",
            "signatures":[
                {"source":"ParentCompany","target":"Industry"},
                {"source":"SubsidiaryCompany","target":"Industry"},
            ],
        },
    },
}

RELATION_SIGNATURES = {
    "AFFILIATED_WITH":[("ParentCompany","ParentCompany","통합 CSV의 top_crno와 affiliate_crno")],
    "HAS_SUBSIDIARY":[("ParentCompany","SubsidiaryCompany","모기업과 종속기업 매칭")],
    "LOCATED_IN":[
        ("ParentCompany","Region","모기업 또는 계열회사의 지역"),
        ("SubsidiaryCompany","Region","종속기업의 지역"),
    ],
    "IN_INDUSTRY":[
        ("ParentCompany","Industry","모기업 또는 계열회사의 SIC 업종"),
        ("SubsidiaryCompany","Industry","종속기업의 주요 사업 내용"),
    ],
}

def build_ontology_block(signatures):
    lines = ["[허용된 관계 시그니처]"]
    for relation, signature_list in signatures.items():
        for source_type, target_type, criterion in signature_list:
            lines.append(f"- {relation}: ({source_type}) -> ({target_type}) # {criterion}")
    return "\n".join(lines)

print(json.dumps(ONTOLOGY_SCHEMA, ensure_ascii=False, indent=2))
print(build_ontology_block(RELATION_SIGNATURES))


{
  "nodes": {
    "ParentCompany": {
      "description": "계열관계 또는 종속관계의 기준이 되는 법인 기업",
      "properties": {
        "crno": {
          "type": "string",
          "description": "법인등록번호"
        },
        "name": {
          "type": "string",
          "description": "기업명"
        },
        "address": {
          "type": "string",
          "description": "기업 주소"
        }
      }
    },
    "SubsidiaryCompany": {
      "description": "모기업과 종속관계를 가지는 기업",
      "properties": {
        "name": {
          "type": "string",
          "description": "종속기업명"
        },
        "name_norm": {
          "type": "string",
          "description": "기업명 매칭을 위한 정규화 이름"
        },
        "address": {
          "type": "string",
          "description": "종속기업 주소"
        },
        "business_content": {
          "type": "string",
          "description": "종속기업의 주요 사업 내용"
        },
        "domestic": {
          "type": "string",
          "description": "국내 또는 해외 구분"
        }
      }
  

In [2]:
# CSV의 빈 값과 HTML 엔티티를 제거하고 공백을 표준화한다.
def text(value):
    if value is None:
        return ""
    value = str(value)
    if value.lower() in {"nan","none","nat"}:
        return ""
    value = html.unescape(value).replace("&cr;", " ")
    return re.sub(r"\s+", " ", value).strip()

# 법인등록번호에서 숫자만 남기고, 0만 있는 값은 빈 값으로 처리한다.
def clean_crno(value):
    digits = re.sub(r"\D", "", text(value))
    return "" if not digits or set(digits) == {"0"} else digits

# 세미콜론 등으로 묶인 복수 crno를 각각의 ID로 분리한다.
def split_ids(value):
    value = text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*,\s*|\s*\|\s*|\s+", value)
    return list(dict.fromkeys(x for x in (clean_crno(v) for v in parts) if x))

# 지역과 같은 범주형 값을 각각의 값으로 나눈다.
def split_values(value):
    value = text(value)
    if not value:
        return []
    parts = re.split(r"\s*;\s*|\s*\|\s*|\r?\n", value)
    return list(dict.fromkeys(x for x in (text(v) for v in parts) if x))

# 이름과 주소를 비교할 때 공백과 특수문자 차이를 제거한 정규화 키를 만든다.
def normalize_key(value):
    value = text(value).lower()
    return re.sub(r"[^0-9a-z가-힣]", "", value)

# crno는 문자열로 읽어야 앞자리 0이 유지된다. 빈 칸은 모두 빈 문자열로 바꾼다.
corp = pd.read_csv(CORP_PATH, dtype=str, encoding="utf-8-sig").fillna("")
integrated = pd.read_csv(INTEGRATED_PATH, dtype=str, encoding="utf-8-sig", keep_default_na=False)

# evidence용: start_col부터 end_col까지의 원천 값을 컬럼 순서대로, 빈 값은 빼고 공백 하나로 잇는다.
# 검증 코드가 원천 행을 만드는 방식과 같아서, 이 문자열은 원천 행에 그대로 들어 있다.
# 두 값 사이의 컬럼까지 포함해야 하는 이유: 떨어진 두 컬럼 값만 붙이면 원천 행에 없는 문자열이 된다.
INTEGRATED_COLUMNS = list(integrated.columns)

def row_span(row, start_col, end_col):
    i = INTEGRATED_COLUMNS.index(start_col)
    j = INTEGRATED_COLUMNS.index(end_col)
    return " ".join(row[c] for c in INTEGRATED_COLUMNS[i:j + 1] if row[c] != "")

corp["crno"] = corp["crno"].map(clean_crno)
corp = corp[corp["crno"].ne("")].drop_duplicates("crno", keep="last")
CORP_IDS = set(corp["crno"])
CORP_INFO = corp.set_index("crno", drop=False).to_dict("index")

# Industry 노드 이름은 통합 CSV의 SIC 업종명(top_sicNm, affiliate_sicNm)을 기준으로 만든다.
# 종속기업의 subsidiary_bizCtt는 SIC 업종명과 정규화 키가 같을 때만 Industry 노드와 연결한다.
# 그 밖의 사업 내용은 SubsidiaryCompany의 business_content 속성에만 남긴다.
INDUSTRY_NAME_BY_KEY = {}
for column in ("top_sicNm", "affiliate_sicNm"):
    for raw_value in integrated[column]:
        industry_name = text(raw_value)
        industry_key = normalize_key(industry_name)
        if industry_key:
            INDUSTRY_NAME_BY_KEY.setdefault(industry_key, industry_name)


nodes = {}
triples = []

# 반복된 노드는 ID를 기준으로 하나로 합친다. 빈 속성은 다른 행의 값으로 보완한다.
def upsert_node(node_id, node_type, properties):
    record = nodes.setdefault(
        (node_id, node_type),
        {"id":node_id, "type":node_type, "properties":{}},
    )
    for key, value in properties.items():
        value = text(value)
        if value and not record["properties"].get(key):
            record["properties"][key] = value
    return node_id

# 기업개요 CSV에 존재하는 crno만 ParentCompany 노드로 생성한다.
def parent_node(crno, fallback_name="", fallback_address=""):
    if crno not in CORP_IDS:
        return None
    info = CORP_INFO.get(crno, {})
    name = text(info.get("corpNm")) or text(info.get("enpPbanCmpyNm")) or text(fallback_name)
    address = " ".join(
        x for x in [text(info.get("enpBsadr")), text(info.get("enpDtadr"))] if x
    ) or text(fallback_address)
    return upsert_node(
        f"parent_company:{crno}", "ParentCompany",
        {"crno":crno, "name":name, "address":address},
    )

# 트리플과 출처 행 번호를 함께 저장해 추후 검증과 추적이 가능하게 한다.
def emit(subject, subject_type, relation, obj, object_type, row_no, case, evidence=""):
    triples.append({
        "subject":subject, "subject_type":subject_type, "relation":relation,
        "object":obj, "object_type":object_type,
        "source_case":text(case), "source_row":int(row_no),
        "evidence":evidence,  # 원천 행과 글자 그대로 대조하므로 정리하지 않는다
    })

# 필터링한 기업개요의 crno 전체를 ParentCompany 노드의 기준으로 사용한다.
for crno in sorted(CORP_IDS):
    parent_node(crno)

# 통합 CSV를 행 단위로 순회하며 계열관계와 종속관계 트리플을 생성한다.
for row_no, row in integrated.iterrows():
    case = row.get("case", "")
    top_ids = split_ids(row.get("top_crno", ""))
    affiliate_id = clean_crno(row.get("affiliate_crno", ""))

    for top_id in top_ids:
        parent_node(top_id, row.get("top_corpNm",""), row.get("top_addr",""))
    parent_node(affiliate_id, row.get("affiliate_corpNm",""), row.get("affiliate_addr",""))

    if affiliate_id in CORP_IDS:
        for top_id in top_ids:
            if top_id in CORP_IDS and top_id != affiliate_id:
                emit(
                    f"parent_company:{top_id}", "ParentCompany", "AFFILIATED_WITH",
                    f"parent_company:{affiliate_id}", "ParentCompany",
                    row_no, case, row_span(row, "top_corpNm", "affiliate_corpNm"),
                )

    # case3은 affiliate가 종속기업의 직접 모기업이므로 affiliate를 우선한다. 없으면 top_crno를 사용한다.
    subsidiary_name = text(row.get("subsidiary_name",""))
    subsidiary_address = text(row.get("subsidiary_addr",""))
    if subsidiary_name or subsidiary_address:
        # 모기업 crno를 ID에 넣지 않고 종속기업을 전역 하나의 노드로 구분한다.
        name_norm = normalize_key(row.get("name_norm","")) or normalize_key(subsidiary_name)
        address_norm = normalize_key(subsidiary_address)
        domestic_norm = normalize_key(row.get("domestic",""))
        business_norm = normalize_key(row.get("subsidiary_bizCtt",""))
        # 이름과 주소가 모두 있으면 같은 종속기업으로 합치고, 주소가 없으면 보조 키를 추가한다.
        if name_norm and address_norm:
            subsidiary_key = f"name:{name_norm}|address:{address_norm}"
        elif name_norm and (domestic_norm or business_norm):
            subsidiary_key = f"name:{name_norm}|fallback:{domestic_norm}|{business_norm}"
        elif name_norm:
            subsidiary_key = f"name:{name_norm}|row:{row_no}"
        else:
            subsidiary_key = f"address:{address_norm}|row:{row_no}" if address_norm else ""
        immediate_parents = [affiliate_id] if affiliate_id else top_ids
        if subsidiary_key:
            subsidiary_id = f"subsidiary:{subsidiary_key}"
            upsert_node(
                subsidiary_id, "SubsidiaryCompany",
                {
                    "name":subsidiary_name,
                    "name_norm":name_norm,
                    "address":subsidiary_address,
                    "business_content":row.get("subsidiary_bizCtt",""),
                    "domestic":row.get("domestic",""),
                },
            )
            # 직접 모기업이 affiliate면 affiliate_corpNm부터, top이면 top_corpNm부터 종속기업명까지 잇는다.
            parent_name_col = "affiliate_corpNm" if affiliate_id else "top_corpNm"
            for parent_id in dict.fromkeys(immediate_parents):
                if parent_id in CORP_IDS:
                    emit(
                        f"parent_company:{parent_id}", "ParentCompany", "HAS_SUBSIDIARY",
                        subsidiary_id, "SubsidiaryCompany",
                        row_no, case, row_span(row, parent_name_col, "subsidiary_name"),
                    )

            for region in split_values(row.get("subsidiary_region","")):
                region_key = normalize_key(region)
                if region_key:
                    region_id = f"region:{region_key}"
                    upsert_node(region_id, "Region", {"name":region})
                    emit(
                        subsidiary_id, "SubsidiaryCompany", "LOCATED_IN",
                        region_id, "Region", row_no, case,
                        row_span(row, "subsidiary_name", "subsidiary_region"),
                    )

            # 종속기업의 사업 내용이 SIC 업종명과 정규화 키로 일치할 때만 IN_INDUSTRY를 만든다.
            # evidence는 매칭된 SIC 업종명이 아니라 원천 행의 종속기업명~사업 내용 원본이다.
            business = text(row.get("subsidiary_bizCtt",""))
            business_key = normalize_key(business)
            standard_industry = INDUSTRY_NAME_BY_KEY.get(business_key)
            if standard_industry:
                industry_id = f"industry:{business_key}"
                upsert_node(industry_id, "Industry", {"name":standard_industry})
                emit(
                    subsidiary_id, "SubsidiaryCompany", "IN_INDUSTRY",
                    industry_id, "Industry", row_no, case,
                    row_span(row, "subsidiary_name", "subsidiary_bizCtt"),
                )

    # top_crno가 하나인 행에서만 top_region을 해당 기업에 연결한다. 여러 crno가 함께 있는 행은 제외한다.
    if len(top_ids) == 1 and top_ids[0] in CORP_IDS:
        for region in split_values(row.get("top_region","")):
            region_key = normalize_key(region)
            if region_key:
                region_id = f"region:{region_key}"
                upsert_node(region_id, "Region", {"name":region})
                emit(
                    f"parent_company:{top_ids[0]}", "ParentCompany", "LOCATED_IN",
                    region_id, "Region", row_no, case,
                    row_span(row, "top_corpNm", "top_region"),
                )

    if affiliate_id in CORP_IDS:
        for region in split_values(row.get("affiliate_region","")):
            region_key = normalize_key(region)
            if region_key:
                region_id = f"region:{region_key}"
                upsert_node(region_id, "Region", {"name":region})
                emit(
                    f"parent_company:{affiliate_id}", "ParentCompany", "LOCATED_IN",
                    region_id, "Region", row_no, case,
                    row_span(row, "affiliate_corpNm", "affiliate_region"),
                )

    # 모기업과 계열회사의 SIC 업종을 Industry 노드와 연결한다.
    # evidence는 기업명 컬럼부터 업종 컬럼까지의 원천 값이다.
    for parent_id, value, evidence in (
        [(x, row.get("top_sicNm",""), row_span(row, "top_corpNm", "top_sicNm")) for x in top_ids]
        + [(affiliate_id, row.get("affiliate_sicNm",""), row_span(row, "affiliate_corpNm", "affiliate_sicNm"))]
    ):
        value = text(value)
        value_key = normalize_key(value)
        if parent_id in CORP_IDS and value_key:
            industry_id = f"industry:{value_key}"
            upsert_node(industry_id, "Industry", {"name":value})
            emit(
                f"parent_company:{parent_id}", "ParentCompany", "IN_INDUSTRY",
                industry_id, "Industry", row_no, case, evidence,
            )

print(f"읽은 기업개요 행 수={len(corp):,}, 통합 데이터 행 수={len(integrated):,}")
print(f"중복 제거 전 노드 수={len(nodes):,}, 중복 제거 전 트리플 수={len(triples):,}")


읽은 기업개요 행 수=862, 통합 데이터 행 수=17,398
중복 제거 전 노드 수=8,574, 중복 제거 전 트리플 수=86,266


In [3]:
# 관계의 주체 및 객체 종류가 온톨로지 시그니처와 일치하는지 확인한다.
def check_signature(triple):
    allowed = RELATION_SIGNATURES.get(triple["relation"], [])
    actual = (triple["subject_type"], triple["object_type"])
    valid_pairs = {(source, target) for source, target, _ in allowed}
    if actual not in valid_pairs:
        return False, f"{triple['relation']}: {actual} is not allowed"
    return True, ""

# 생성한 트리플을 표 형식으로 변환한다.
triples_df = pd.DataFrame(triples)
if triples_df.empty:
    triples_df = pd.DataFrame(columns=[
        "subject","subject_type","relation","object","object_type",
        "source_case","source_row","evidence",
    ])
else:
    triples_df = triples_df.drop_duplicates(
        subset=["subject","relation","object"], keep="first"
    ).reset_index(drop=True)

# 허용되지 않은 관계 시그니처가 발견되면 출력 전에 즉시 확인한다.
errors = []
for record in triples_df.to_dict("records"):
    valid, reason = check_signature(record)
    if not valid:
        errors.append(reason)
assert not errors, errors[:10]

# 노드 ID, 노드 종류, 속성을 후속 적재에 쓰기 쉽도록 표 형식으로 변환한다.
nodes_df = pd.DataFrame([
    {"id":value["id"], "type":value["type"], **value["properties"]}
    for value in nodes.values()
])
if not nodes_df.empty:
    nodes_df = nodes_df.sort_values(["type","id"]).reset_index(drop=True)

# 관계 유형별 최종 트리플 개수를 요약해 검수 결과를 확인한다.
summary = (
    triples_df.groupby(["relation","subject_type","object_type"], dropna=False)
    .size().reset_index(name="count")
    if not triples_df.empty
    else pd.DataFrame(columns=["relation","subject_type","object_type","count"])
)

# 교안의 JSONL 파일처럼 한 줄에 노드 혹은 트리플 하나만 저장한다.
# 노드 JSONL은 properties 안에 속성을 넣고, 트리플 JSONL은 subject/object와 노드 종류를 함께 저장한다.
NODE_JSONL_OUTPUT = CLEAN_DIR / "기업관계_노드.jsonl"
TRIPLE_JSONL_OUTPUT = CLEAN_DIR / "기업관계_트리플.jsonl"

with NODE_JSONL_OUTPUT.open("w", encoding="utf-8", newline="\n") as file:
    for node in nodes.values():
        file.write(json.dumps(node, ensure_ascii=False) + "\n")

with TRIPLE_JSONL_OUTPUT.open("w", encoding="utf-8", newline="\n") as file:
    for triple in triples_df.to_dict("records"):
        file.write(json.dumps(triple, ensure_ascii=False) + "\n")

print(f"노드 JSONL 저장 완료: {NODE_JSONL_OUTPUT}")
print(f"트리플 JSONL 저장 완료: {TRIPLE_JSONL_OUTPUT}")


노드 JSONL 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_노드.jsonl
트리플 JSONL 저장 완료: c:\Users\Playdata\OneDrive\Desktop\mle-01-p2-team3\data\clean\기업관계_트리플.jsonl


In [19]:
triples

[{'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'AFFILIATED_WITH',
  'object': 'parent_company:1101110014764',
  'object_type': 'ParentCompany',
  'source_case': '1',
  'source_row': 0,
  'evidence': '롯데쇼핑(주) 서울 서울특별시 중구  남대문로 81 (소공동) 백화점 1101110014764 롯데건설주식회사'},
 {'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'LOCATED_IN',
  'object': 'region:서울',
  'object_type': 'Region',
  'source_case': '1',
  'source_row': 0,
  'evidence': '롯데쇼핑(주) 서울'},
 {'subject': 'parent_company:1101110014764',
  'subject_type': 'ParentCompany',
  'relation': 'LOCATED_IN',
  'object': 'region:서울',
  'object_type': 'Region',
  'source_case': '1',
  'source_row': 0,
  'evidence': '롯데건설주식회사 서울'},
 {'subject': 'parent_company:1101110000086',
  'subject_type': 'ParentCompany',
  'relation': 'IN_INDUSTRY',
  'object': 'industry:백화점',
  'object_type': 'Industry',
  'source_case': '1',
  'source_row': 0,
  'evidence': '롯데쇼핑(

In [4]:
# [제공 코드] Neo4j 연결: 이 셀은 실행만 하세요(내용은 이해하지 않아도 됩니다).
# - .env 의 NEO4J_URI/NEO4J_USER/NEO4J_PASSWORD 로 데이터베이스에 연결합니다.
# - run_cypher("쿼리", 파라미터=값) 가 결과를 dict 리스트로 돌려줍니다. 이 헬퍼로 Cypher 를 실행합니다.
# - 반드시 "실습 전용" 데이터베이스에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

# 1) 접속 정보 읽기: .env 의 키=값을 환경변수로 올려 둔다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# os.getenv(키, 기본값): .env 를 못 읽어도 에러가 아니라 이 기본값으로 조용히 넘어간다.
# 그러니 이 셀 마지막 줄에 찍히는 주소가 "실습 전용 DB" 가 맞는지 눈으로 꼭 확인한다
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북과 데이터베이스를 잇는 통로를 하나 열어 둔다(노트북이 끝날 때까지 재사용)
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 지금 바로 접속을 시험한다. 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    # session 은 쿼리를 실행하는 단위. with 블록을 벗어나면 알아서 닫힌다
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 그 키는 RETURN 에 적은 별칭이 된다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

Neo4j 연결: bolt://localhost:7687


In [3]:
# [제공 코드] 실습 전용 DB 초기화: 이 데이터베이스를 통째로 비웁니다.
# ⚠️ 가리는 것 없이 **노드·관계·제약을 전부** 지웁니다.
run_cypher("MATCH (n) DETACH DELETE n")   # DETACH 는 노드에 붙은 관계까지 함께 지웁니다
# 제약조건은 노드를 지워도 남습니다. 이름을 조회해 하나씩 DROP 합니다
for _c in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _c["name"] + " IF EXISTS")
print("초기화 완료:", NEO4J_URI, "· 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

NameError: name 'NEO4J_URI' is not defined

In [7]:
import json
from collections import defaultdict
from pathlib import Path

# JSONL 파일 위치
# 커널의 작업 디렉터리가 notebooks/ 일 수도 있으므로 프로젝트 루트를 찾아 절대경로로 만든다.
NODE_FILENAME = "기업관계_노드.jsonl"
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "clean" / NODE_FILENAME).exists()),
    Path.cwd(),
)
CLEAN_DIR = PROJECT_ROOT / "data" / "clean"
NODE_JSONL_PATH = CLEAN_DIR / NODE_FILENAME
TRIPLE_JSONL_PATH = CLEAN_DIR / "기업관계_트리플.jsonl"

BATCH_SIZE = 1000

NODE_TYPES = {
    "ParentCompany",
    "SubsidiaryCompany",
    "Region",
    "Industry",
}

ALLOWED_SIGNATURES = {
    ("AFFILIATED_WITH", "ParentCompany", "ParentCompany"),
    ("HAS_SUBSIDIARY", "ParentCompany", "SubsidiaryCompany"),
    ("LOCATED_IN", "ParentCompany", "Region"),
    ("LOCATED_IN", "SubsidiaryCompany", "Region"),
    ("IN_INDUSTRY", "ParentCompany", "Industry"),
    ("IN_INDUSTRY", "SubsidiaryCompany", "Industry"),
}


def read_jsonl(path):
    """JSONL 파일을 한 줄씩 읽어 dict 리스트로 반환한다."""
    with path.open("r", encoding="utf-8") as file:
        return [json.loads(line) for line in file if line.strip()]


def run_in_batches(query, rows, batch_size=BATCH_SIZE):
    """많은 데이터를 한 번에 보내지 않고 일정 개수씩 나누어 적재한다."""
    for start in range(0, len(rows), batch_size):
        batch = rows[start:start + batch_size]
        run_cypher(query, rows=batch)


# 1. 노드 JSONL 읽기
node_rows = read_jsonl(NODE_JSONL_PATH)

nodes_by_type = defaultdict(list)

for row in node_rows:
    node_type = row["type"]

    if node_type not in NODE_TYPES:
        raise ValueError(f"허용되지 않은 노드 타입: {node_type}")

    nodes_by_type[node_type].append(row)


# 2. 노드 ID 유일성 제약조건 생성
# 노드의 id는 기업의 crno 또는 종속기업의 합성 ID이다.
for node_type in NODE_TYPES:
    constraint_name = f"{node_type.lower()}_id_unique"

    query = f"""
    CREATE CONSTRAINT {constraint_name} IF NOT EXISTS
    FOR (n:{node_type})
    REQUIRE n.id IS UNIQUE
    """

    run_cypher(query)


# 3. 노드 적재
# label은 파라미터로 전달할 수 없으므로 노드 타입별로 쿼리를 생성한다.
for node_type, rows in nodes_by_type.items():
    query = f"""
    UNWIND $rows AS row
    MERGE (n:{node_type} {{id: row.id}})
    SET n += row.properties
    SET n.id = row.id
    """

    run_in_batches(query, rows)


# 4. 트리플 JSONL 읽기
triple_rows = read_jsonl(TRIPLE_JSONL_PATH)

triples_by_signature = defaultdict(list)

for row in triple_rows:
    signature = (
        row["relation"],
        row["subject_type"],
        row["object_type"],
    )

    if signature not in ALLOWED_SIGNATURES:
        raise ValueError(f"허용되지 않은 관계 시그니처: {signature}")

    triples_by_signature[signature].append(row)


# 5. 관계 적재
# 관계 타입과 노드 label은 허용된 시그니처에서 검증된 값만 쿼리에 삽입한다.
for (relation, subject_type, object_type), rows in triples_by_signature.items():
    query = f"""
    UNWIND $rows AS row

    MATCH (subject:{subject_type} {{id: row.subject}})
    MATCH (object:{object_type} {{id: row.object}})

    MERGE (subject)-[r:{relation}]->(object)

    SET r.source_case = row.source_case,
        r.source_row = row.source_row,
        r.evidence = row.evidence
    """

    run_in_batches(query, rows)


print(f"노드 적재 대상: {len(node_rows):,}개")
print(f"트리플 적재 대상: {len(triple_rows):,}개")

노드 적재 대상: 8,574개
트리플 적재 대상: 19,160개


## 생성되는 관계

(ParentCompany, AFFILIATED_WITH, ParentCompany)
(ParentCompany, HAS_SUBSIDIARY, SubsidiaryCompany)
(ParentCompany, LOCATED_IN, Region)
(SubsidiaryCompany, LOCATED_IN, Region)
(ParentCompany, IN_INDUSTRY, Industry)
(SubsidiaryCompany, IN_INDUSTRY, Industry)

종속기업에 실제 crno가 제공되면 합성 ID를 해당 crno 기반 ID로 교체할 수 있다. 이름과 주소가 모두 비어 있는 종속기업은 노드로 만들지 않는다.


In [5]:
# Neo4j Aura(클라우드) 연결: 아래 뉴스 벡터 노드·관계 적재에만 쓴다.
# - .env 의 AURA_URI/AURA_USER/AURA_PASSWORD 를 쓴다.
# - 로컬용 driver / run_cypher 는 건드리지 않고 aura_driver / run_aura 를 따로 만든다.
#   그래서 위의 노드·트리플 적재는 로컬로, 아래 뉴스 적재는 클라우드로 간다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase

load_dotenv(".env", override=True)
load_dotenv("../.env", override=True)

AURA_URI = os.getenv("AURA_URI")
AURA_USER = os.getenv("AURA_USER", "neo4j")
AURA_PASSWORD = os.getenv("AURA_PASSWORD")

if not AURA_URI or not AURA_PASSWORD:
    raise RuntimeError(".env 에 AURA_URI / AURA_PASSWORD 를 넣으세요.")


def connect_aura(uri, user, password):
    """Aura 에 붙는다. 막히는 지점이 두 군데라 각각 한 번씩 더 시도한다.

    - 주소: 사내망이 TLS 를 가로채면 인증서 검증에서 막힌다 -> neo4j+ssc 로 재시도
    - 계정: 최근 만든 인스턴스는 사용자명이 neo4j 가 아니라 인스턴스 ID 다 -> ID 로 재시도
    """
    host = uri.split("://")[-1]
    instance_id = host.split(".")[0]              # e8fbc6d2.databases.neo4j.io -> e8fbc6d2

    uris = [uri]
    if not uri.startswith("neo4j+ssc"):
        uris.append(f"neo4j+ssc://{host}")        # ssc = 자체 서명 인증서 허용(검증 생략)
    users = list(dict.fromkeys([user, instance_id]))

    last_error = None
    for candidate_uri in uris:
        for candidate_user in users:
            try:
                new_driver = GraphDatabase.driver(candidate_uri, auth=(candidate_user, password))
                new_driver.verify_connectivity()
                return new_driver, candidate_uri, candidate_user
            except Exception as error:
                last_error = error
    raise last_error


aura_driver, AURA_URI_USED, AURA_USER_USED = connect_aura(AURA_URI, AURA_USER, AURA_PASSWORD)


def run_aura(query, **params):
    """Aura 에서 Cypher 실행 -> 결과를 dict 리스트로 반환(run_cypher 와 같은 모양, 접속 대상만 다르다)."""
    with aura_driver.session() as session:
        return [record.data() for record in session.run(query, **params)]


# 뉴스 노드가 올라갈 곳이다. 주소와 기존 노드 수를 눈으로 확인한다
print("접속 대상: Aura(클라우드)")
print("  URI     :", AURA_URI_USED)
print("  사용자  :", AURA_USER_USED)
print("  노드    :", run_aura("MATCH (n) RETURN count(n) AS c")[0]["c"])
print("  관계    :", run_aura("MATCH ()-[r]->() RETURN count(r) AS c")[0]["c"])
for row in run_aura("CALL dbms.components() YIELD name, versions, edition RETURN name, versions, edition"):
    print("  버전    :", row["name"], row["versions"], row["edition"])


접속 대상: Aura(클라우드)
  URI     : neo4j+ssc://e8fbc6d2.databases.neo4j.io
  사용자  : e8fbc6d2
  노드    : 8574
  관계    : 19160
  버전    : Neo4j Kernel ['5.27-aura'] enterprise
  버전    : Cypher ['5', '25'] 


## 뉴스 벡터 노드 스키마

기사 한 건이 노드 하나다. `news_articles.jsonl` 의 `record_id`(sha256 64자)를 노드 id 로 쓴다.
같은 기업의 기사가 여러 건이라 기업 단위로 묶지 않고 기사별로 따로 만든다.

**노드 `News`**

| 속성 | 원본 필드 | 설명 |
|---|---|---|
| `id` | `record_id` | 기사 고유 id. 재적재해도 중복되지 않는 열쇠 |
| `crno` | `company_crno` | 모기업 법인등록번호. 관계를 잇는 열쇠 |
| `date` | `published_at` | 발행일. RFC 2822 문자열을 `YYYY-MM-DD` 로 바꾼다 |
| `title` | `title` | 기사 제목 |
| `summary` | `description` | 기사 요약 |
| `publisher` | `publisher` | 언론사 도메인 |
| `url` | `original_url` | 원문 주소 |
| `embedding` | (계산) | `title + summary` 를 임베딩한 768차원 벡터 |

**관계**

```
(News) -[:RELATED_TO]-> (ParentCompany)
```

`News.crno` 와 `ParentCompany.crno` 를 맞춰 잇는다. 기사 380건이 기업 84곳에 걸려 있고,
84곳 모두 `ParentCompany` 노드에 존재한다.

**임베딩 규칙**

- 모델 `text-embedding-3-large`, `dimensions=768` 로 잘라 받는다
- 대상 텍스트는 `title + " " + summary`. 제목에만 있는 고유명사를 놓치지 않으려는 것이다
- 유사도 함수는 코사인. 벡터 인덱스도 같은 값(768, cosine)으로 맞춰야 한다


In [6]:
# 뉴스 원본을 읽어 News 노드에 넣을 모양으로 정리한다.
import json
from email.utils import parsedate_to_datetime
from pathlib import Path

NEWS_FILENAME = "최종_뉴스기사.jsonl"
NEWS_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "data" / "clean" / NEWS_FILENAME).exists()),
    Path.cwd(),
)
NEWS_PATH = NEWS_ROOT / "data" / "clean" / NEWS_FILENAME


def to_date(value):
    """'Thu, 17 Sep 2026 13:36:00 +0900' -> '2026-09-17'. 못 읽으면 빈 문자열."""
    try:
        return parsedate_to_datetime(value).date().isoformat()
    except (TypeError, ValueError):
        return ""


with NEWS_PATH.open(encoding="utf-8") as file:
    articles = [json.loads(line) for line in file if line.strip()]

news_rows = [
    {
        "id": article["record_id"],
        "crno": article["company_crno"],
        "date": to_date(article.get("published_at")),
        "title": (article.get("title") or "").strip(),
        "summary": (article.get("description") or "").strip(),
        "publisher": (article.get("publisher") or "").strip(),
        "url": (article.get("original_url") or "").strip(),
    }
    for article in articles
]

# id 가 겹치면 뒤에서 MERGE 가 기사를 덮어쓴다. 먼저 확인한다.
assert len({row["id"] for row in news_rows}) == len(news_rows), "record_id 중복"

print(f"기사 {len(news_rows):,}건 / 기업 {len({r['crno'] for r in news_rows}):,}곳")
print(f"날짜 범위 {min(r['date'] for r in news_rows)} ~ {max(r['date'] for r in news_rows)}")
print(news_rows[0])


기사 380건 / 기업 84곳
날짜 범위 2020-09-24 ~ 2026-09-17
{'id': 'fcafd48df4d3121f0b681eee26dc79e7628227fea6126e0602b9f1b7263226d2', 'crno': '1101110085450', 'date': '2026-09-17', 'title': "현대자동차직업전문학교, '과정평가형 자동차정비산업기사' 과정 10월 개...", 'summary': '대전 현대자동차직업전문학교(이사장 유성식)가 미래 자동차정비산업기사 인재 양성을 위해 대전RSC와 협력해 자동차 기업 및 지역 유관기관과의 산학협력 체계를 확대하고 있다. 대전 현대직업전문학교는 대전RSC와 함께...', 'publisher': 'mhns.co.kr', 'url': 'https://www.mhns.co.kr/news/articleView.html?idxno=760782'}


In [7]:
# 이을 상대가 Aura 그래프에 있는지 먼저 확인한다. 없는 crno 는 관계가 안 생긴다.
graph_crnos = {
    row["crno"]
    for row in run_aura("MATCH (p:ParentCompany) RETURN p.crno AS crno")
}
news_crnos = {row["crno"] for row in news_rows}

print(f"기사에 나오는 기업 {len(news_crnos):,}곳")
print(f"  ParentCompany 에 있음 : {len(news_crnos & graph_crnos):,}곳")
print(f"  그래프에 없음        : {len(news_crnos - graph_crnos):,}곳")

if news_crnos - graph_crnos:
    print("  ->", sorted(news_crnos - graph_crnos)[:10])


기사에 나오는 기업 84곳
  ParentCompany 에 있음 : 84곳
  그래프에 없음        : 0곳


In [8]:
# 임베딩 준비: title + summary 를 768차원으로 받는다.
# 한 번 계산한 것은 파일에 저장해 두고 다시 쓴다(재실행할 때 다시 부르지 않으려고).
import pickle

from langchain_openai import OpenAIEmbeddings

EMBED_MODEL = "text-embedding-3-large"
EMBED_DIM = 768                    # 3072차원으로 나오는 모델을 768차원으로 잘라 받는다
EMB_FILE = NEWS_ROOT / "data" / "news_emb_cache.pkl"
EMB_CACHE = pickle.loads(EMB_FILE.read_bytes()) if EMB_FILE.exists() else {}

embedder = OpenAIEmbeddings(model=EMBED_MODEL, dimensions=EMBED_DIM)


def embed_texts(texts):
    """문서 리스트 -> 768차원 임베딩 리스트. 저장된 것은 그대로 쓰고 없는 것만 부른다."""
    new = [text for text in texts if text not in EMB_CACHE]
    if new:
        EMB_CACHE.update(zip(new, embedder.embed_documents(new)))
        EMB_FILE.parent.mkdir(parents=True, exist_ok=True)
        EMB_FILE.write_bytes(pickle.dumps(EMB_CACHE))
    return [EMB_CACHE[text] for text in texts]


def embed_query(text):
    """질문 한 문장 -> 768차원 임베딩. 질문은 저장하지 않는다."""
    return embedder.embed_query(text)


# 제목과 요약을 이어 붙인다. 제목에만 있는 회사명·제품명을 놓치지 않으려는 것이다.
embedding_texts = [f"{row['title']} {row['summary']}".strip() for row in news_rows]

print("임베딩 모델:", EMBED_MODEL, f"({EMBED_DIM}차원)")
print("저장된 문서:", len(EMB_CACHE), "건 / 이번에 필요한 문서:", len(set(embedding_texts)), "건")
print("예시:", embedding_texts[0][:80])


임베딩 모델: text-embedding-3-large (768차원)
저장된 문서: 377 건 / 이번에 필요한 문서: 377 건
예시: 현대자동차직업전문학교, '과정평가형 자동차정비산업기사' 과정 10월 개... 대전 현대자동차직업전문학교(이사장 유성식)가 미래 자동차정비산업기사


In [9]:
# Aura 에 News 노드를 만들고 ParentCompany 와 잇는다.
# 재실행해도 중복이 생기지 않도록 id 에 제약을 걸고 MERGE 로 올린다.
run_aura("""
CREATE CONSTRAINT news_id_unique IF NOT EXISTS
FOR (n:News) REQUIRE n.id IS UNIQUE
""")

BATCH_SIZE = 1000   # 클라우드는 왕복마다 네트워크 지연이 붙어 한 번에 많이 보낸다


def in_batches(query, rows, batch_size=BATCH_SIZE):
    for start in range(0, len(rows), batch_size):
        run_aura(query, rows=rows[start:start + batch_size])


# 1) 노드 올리기
in_batches("""
UNWIND $rows AS row
MERGE (n:News {id: row.id})
SET n.crno = row.crno,
    n.date = row.date,
    n.title = row.title,
    n.summary = row.summary,
    n.publisher = row.publisher,
    n.url = row.url
""", news_rows)

# 2) 모기업과 잇기. crno 가 그래프에 있는 기사만 관계가 생긴다
in_batches("""
UNWIND $rows AS row
MATCH (n:News {id: row.id})
MATCH (p:ParentCompany {crno: row.crno})
MERGE (n)-[:RELATED_TO]->(p)
""", news_rows)

# 3) 임베딩 붙이기. 벡터는 전용 프로시저로 저장해야 공간을 덜 쓴다
vectors = embed_texts(embedding_texts)
in_batches("""
UNWIND $rows AS row
MATCH (n:News {id: row.id})
CALL db.create.setNodeVectorProperty(n, 'embedding', row.vec)
""", [{"id": row["id"], "vec": vector}
      for row, vector in zip(news_rows, vectors)])

print("News 노드      :", run_aura("MATCH (n:News) RETURN count(n) AS c")[0]["c"])
print("RELATED_TO 관계:", run_aura("MATCH (:News)-[r:RELATED_TO]->(:ParentCompany) RETURN count(r) AS c")[0]["c"])
print("임베딩 붙은 노드:", run_aura("MATCH (n:News) WHERE n.embedding IS NOT NULL RETURN count(n) AS c")[0]["c"])
print("관계 없는 기사  :", run_aura("MATCH (n:News) WHERE NOT (n)-[:RELATED_TO]->() RETURN count(n) AS c")[0]["c"])


News 노드      : 380
RELATED_TO 관계: 380
임베딩 붙은 노드: 380
관계 없는 기사  : 0


다음 단계는 벡터 인덱스다. 노드에 벡터를 담는 것까지가 검색 준비의 절반이고,
인덱스를 세워야 데이터베이스가 가까운 것을 대신 찾아 준다. 차원과 유사도 함수를
임베딩과 같은 값(768, cosine)으로 맞춰야 한다.


In [10]:
# Aura 에 뉴스 벡터 인덱스를 만든다. 차원·유사도 함수는 임베딩과 같은 값(768, cosine)으로 맞춘다.
run_aura("""
CREATE VECTOR INDEX news_vec IF NOT EXISTS
FOR (n:News) ON n.embedding
OPTIONS {indexConfig: {
  `vector.dimensions`: 768,
  `vector.similarity_function`: 'cosine'
}}
""")
run_aura("CALL db.awaitIndexes()")   # 인덱스가 다 설 때까지 기다린다

# state 가 ONLINE 이어야 검색에 쓰인다
for row in run_aura("""SHOW VECTOR INDEXES YIELD name, state, labelsOrTypes, properties, options
                       RETURN name, state, labelsOrTypes, properties, options.indexConfig AS config"""):
    print(row["name"], row["state"], row["labelsOrTypes"], row["properties"])
    print("   차원:", row["config"]["vector.dimensions"], "/ 유사도:", row["config"]["vector.similarity_function"])


news_vec ONLINE ['News'] ['embedding']
   차원: 768 / 유사도: COSINE


In [ ]:
# 뉴스 벡터 검색기: 질문 -> 임베딩 -> 벡터 인덱스에서 가장 가까운 기사 top_k 건.
# Aura 가 5.27 이라 교안의 SEARCH 절(2026.01 이상)은 못 쓴다. 5.x 에서는 이 프로시저를 쓴다
# (neo4j-graphrag 의 VectorRetriever 도 5.x 서버에서는 속으로 같은 프로시저를 부른다).
#
# score 는 코사인 유사도를 0~1 로 옮긴 값이다: score = (1 + 코사인) / 2
# 그래서 관련 없는 문장도 0.5 근처가 나온다. 원래 코사인(-1~1)도 함께 찍는다.
def search_news(question, top_k=1):
    """질문과 뜻이 가장 가까운 뉴스를 유사도와 함께 돌려준다. 기사가 가리키는 모기업도 붙인다."""
    return run_aura("""
    CALL db.index.vector.queryNodes('news_vec', $k, $q)
    YIELD node, score
    OPTIONAL MATCH (node)-[:RELATED_TO]->(p:ParentCompany)
    RETURN node.title   AS title,
           node.summary AS summary,
           node.date    AS date,
           node.url     AS url,
           p.name       AS company,
           p.crno       AS crno,
           score
    ORDER BY score DESC
    """, k=top_k, q=embed_query(question))


def show_top1(question):
    """top1 한 건을 유사도와 함께 보기 좋게 찍는다."""
    hits = search_news(question, top_k=1)
    if not hits:
        print("검색 결과 없음 (벡터 인덱스 news_vec 이 ONLINE 인지 확인)")
        return None

    hit = hits[0]
    print("질문    :", question)
    print("유사도  :", round(hit["score"], 4), f"(코사인 {2 * hit['score'] - 1:.4f})")
    print("기업    :", hit["company"], f"({hit['crno']})")
    print("날짜    :", hit["date"])
    print("제목    :", hit["title"])
    print("요약    :", (hit["summary"] or "")[:120])
    print("원문    :", hit["url"])
    return hit


top1 = show_top1("반도체 투자 확대 소식")


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=2, column=5, offset=5>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 5, 'line': 2, 'column': 5}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n    CALL db.index.vector.queryNodes('news_vec', $k, $q)\n    YIELD node, score\n    OPTIONAL MATCH (node)-[:RELATED_TO]->(p:ParentCompany)\n    RETURN node.title   AS title,\n           node.summary AS summary,\n           node.date    AS date,\n           node.url     AS url,\n           p.name       AS company,\n           p.crno       A

질문    : 반도체 투자 확대 소식
유사도  : 0.8177 (코사인 0.6354)
기업    : (주)두산 (1101110013774)
날짜    : 2026-09-17
제목    : 두산, AI 반도체 CCL에 9700억 투자...국내·중국 생산능력 확대
요약    : |중앙이코노미뉴스 송태원 기자|두산그룹 사옥. [사진=중앙이코노미뉴스] ㈜두산이 인공지능(AI) 데이터센터 시장 확대에 대응해 하이엔드 동박적층판(CCL) 생산능력을 대폭 늘린다. 국내와 중국에 총 9700억원을..
원문    : https://www.joongangenews.com/news/articleView.html?idxno=548909


: 